In [2]:
import cdflib
import numpy as np
import pandas as pd
import os
import re
from datetime import datetime
data_path = "/home/docker/data/ro-share/omni/omni_cdaweb/hourly/1996/omni2_h0_mrg1hr_19960101_v01.cdf"
cdf = cdflib.CDF(data_path)
var_names = cdf.cdf_info().rVariables
epoch = cdf.varget('Epoch')
epoch = cdflib.cdfepoch.to_datetime(epoch)
BZ_GSM = cdf.varget('BZ_GSM')
BY_GSM = cdf.varget('BY_GSM')
BX_GSE = cdf.varget('BX_GSE')
V = cdf.varget('V')
Pressure = cdf.varget('Pressure')


AE = cdf.varget('AE')
core_params = ['Epoch','BZ_GSM', 'BY_GSM', 'BX_GSE','V', 'Pressure']
data_path_s = "/home/docker/data/ro-share/omni/omni_cdaweb/hourly/2005/omni2_h0_mrg1hr_20050101_v01.cdf"

cdf_s = cdflib.CDF(data_path_s)
var_names = cdf_s.cdf_info().rVariables

yr2 = cdf_s.varget('YR')
day2 = cdf_s.varget('DAY')
hr2 = cdf_s.varget('HR')
kp2 = cdf_s.varget('KP')
AL2 = cdf_s.varget('AL_INDEX')
AE2 = cdf_s.varget('AE')

base_time2 = datetime.fromisoformat("2005-01-01T00:00:00")
utc_times_s = []
for day in range(0, 181):
    for hour in range(0, 24):
        delta = pd.Timedelta(days=day, hours=hour)
        utc_time = base_time2 + delta
        utc_times_s.append(utc_time)
mn_time_s = np.array(utc_times_s)
kp2 = kp2.astype(np.float32)
AL2 = AL2.astype(np.float32)
AE2 = AE2.astype(np.float32)


base_time = datetime.fromisoformat("1996-01-01T00:00:00")
utc_times = []
for day in range(0, 182):
    for hour in range(0, 24):
        delta = pd.Timedelta(days=day, hours=hour)
        utc_time = base_time + delta
        utc_times.append(utc_time)
mn_time = np.array(utc_times)
kp = kp.astype(np.float32)
AL = AL.astype(np.float32)
AE = AE.astype(np.float32)

df = pd.DataFrame(
    {
        'utc': mn_time,
        'kp': kp,
        'AL': AL,
        'AE': AE
    }
)

df2 = pd.DataFrame(
    {
        'utc': mn_time_s,
        'kp': kp2,
        'AL': AL2,
        'AE': AE2
    }
)

apr1 = datetime(1996, 4, 1)

df_after_apr1 = df[df['utc'] > apr1]
# time1 = datetime.fromisoformat("1996-04-01T07:20:00")
# delta = time1 - base_time
# idx_mn = int(delta.total_seconds() // 300 )
# IMF = cdf.varget('IMF')
# proton_density = cdf.varget('proton_density')
# AE_INDEX = cdf.varget('AE_INDEX')

#omni_path = "/home/docker/data/private/AuroraData/omni_real_data/omni_5min/1996/omni_19960101_5min.npy"
#df = pd.DataFrame(np.load(omni_path,allow_pickle=True))
# solar_data = df.values
# data1 = solar_data[::12]

# omni_path_2 = "/home/docker/data/private/AuroraData/omni_real_data/omni_5min/1996/omni_19960201_5min.npy"
# df2 = pd.DataFrame(np.load(omni_path_2,allow_pickle=True))
# solar_data2 = df2.values
# data2 = solar_data2[::12]


# omni_data = []

# omni_data.append(data1)
# omni_data.append(data2)
# omni_data = np.concatenate(omni_data, axis=0)

# data = np.load(omni_path, allow_pickle=True)
# solar_fields = ['Bx', 'By', 'Bz', 'Vx', 'Vy', 'Vz', 'P']
# solar_components = []
# for field in solar_fields:
#     if field in data.dtype.names:
#         field_data = data[field]
#         if field_data.ndim > 1:
#             if field_data.shape[1] > 0:
#                 field_data = field_data[:, 0]
#             else:
#                 field_data = field_data.flatten()
#         solar_components.append(field_data.astype(np.float32))
#     else:
#         logger.warning(f"字段 {field} 不存在，用0填充")
#         solar_components.append(np.zeros(len(data), dtype=np.float32))
        
# solar_data = np.column_stack(solar_components)
# solar_data = solar_data[::12]


# solar_min = solar_data.min(axis=0)
# solar_max = solar_data.max(axis=0)
# solar_range = solar_max - solar_min
# solar_range[solar_range == 0] = 1.0

# print(solar_min)
# print(solar_max)
# print(solar_range)

In [ ]:
import cdflib
import numpy as np
import pandas as pd
import os
import re
# 核心参数列表
core_params = ['Epoch','BZ_GSM', 'BY_GSM', 'BX_GSE','V', 'Pressure']
# 参数合理范围定义
VALID_RANGES = {
    'BZ_GSM': (-600, 600),      # 典型磁场范围
    'BY_GSM': (-600, 600),
    'BX_GSE': (-600, 600),
    'V' : (0,6000), # 合理的太阳风速度范围
    'Pressure': (0, 600),  
}
def extract_core_omni(cdf_file, output_dir=None):
    """
    从CDF文件中提取核心OMNI参数并保存为npy文件
    """
    # 从文件路径中提取文件名（不含扩展名）
    # 从文件名中提取时间信息
    # 匹配格式：omni_hro_5min_YYYYMMDD_vXX
    file_name = os.path.splitext(os.path.basename(cdf_file))[0]
    print(f"处理文件: {file_name}")
    
    time_match = re.search(r'(\d{8})', file_name)
    if time_match:
        date_str = time_match.group(1)
        print(f"提取到日期: {date_str}")
    else:
        date_str = "unknown_date"
        print("警告: 无法从文件名中提取日期")
        
    # 读取CDF文件
    cdf = cdflib.CDF(cdf_file)
    
    # 提取数据
    data_dict = {}
    for param in core_params:
        data_dict[param] = cdf.varget(param)
    
    #cdf.close()
    data_dict['Epoch'] = cdflib.epochs.CDFepoch.to_datetime(data_dict['Epoch'])
    # 计算太阳风总速度 V = sqrt(Vx² + Vy² + Vz²)
    #data_dict['V'] = np.sqrt(data_dict['Vx']**2 + data_dict['Vy']**2 + data_dict['Vz']**2)
    df = pd.DataFrame(
        {
            'utc':data_dict['Epoch'],
            'Bx':data_dict['BX_GSE'],
            'By':data_dict['BY_GSM'],
            'Bz':data_dict['BZ_GSM'],
            'V' :data_dict['flow_speed'],
            'Vx':data_dict['proton_density'],
            'Vy':data_dict['AE_INDEX'],
        }
    )
     # 1. 将超出范围的值设置为NaN
    print("应用数据范围过滤...")
    range_stats = {}
    
    # 对每个参数应用范围过滤
    param_mapping = {
        'Bx': 'BX_GSE',
        'By': 'BY_GSM', 
        'Bz': 'BZ_GSM',
        'V' :   'V',
        'Vx':   'Vx',
        'Vy':   'Vy',
        'Vz':   'Vz',
        'P': 'Pressure'
    }
    
    for df_col, omni_param in param_mapping.items():
        if omni_param in VALID_RANGES:
            min_val, max_val = VALID_RANGES[omni_param]
            original_count = len(df[df_col])
            
            # 标记超出范围的值
            mask = (df[df_col] >= min_val) & (df[df_col] <= max_val)
            invalid_count = np.sum(~mask)
            
            # 将超出范围的值设为NaN
            df.loc[~mask, df_col] = np.nan
            
            range_stats[df_col] = {
                'original': original_count,
                'invalid': invalid_count,
                'percent_invalid': (invalid_count / original_count) * 100
            }
            
            print(f"{df_col}: 过滤掉 {invalid_count} 个异常点 ({range_stats[df_col]['percent_invalid']:.2f}%)")
    # 2. 应用线性插值
    print("\n应用线性插值...")
    
    # 记录插值前的NaN数量
    nan_before = df.isna().sum()
    
    # 对数值列进行线性插值
    numeric_cols = ['Bx', 'By', 'Bz', 'V', 'Vx', 'Vy', 'Vz', 'P']
    for col in numeric_cols:
        # 使用线性插值填充NaN值
        df[col] = df[col].interpolate(method='linear', limit_direction='both')
    
    # 记录插值后的NaN数量
    nan_after = df.isna().sum()
    
    print("\n=== 插值效果 ===")
    for col in numeric_cols:
        filled_count = nan_before[col] - nan_after[col]
        print(f"{col}: 填充了 {filled_count} 个NaN值，剩余 {nan_after[col]} 个NaN")
        if nan_after[col] > 0:
            print(f"边界仍有 {nan_after[col]} 个NaN，使用前向/后向填充")
            df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
    
    
    file = f"omni_{date_str}_1min.npy"
    output_file = os.path.join(output_dir,file)
    structured_array = df.to_records(index=False)
    
    np.save(output_file, structured_array)
    print(f"已保存: {output_file}")

# 使用示例
# if __name__ == "__main__":
#     cdf_file = "/home/docker/data/ro-share/omni/omni_cdaweb/hro_5min/1996/omni_hro_5min_19960101_v01.cdf"  # 替换为你的CDF文件路径
#     output_dir = "/home/docker/data/private/AuroraData/omni_real_data/omni_5min"
#     data = extract_core_omni(cdf_file, output_dir)


In [2]:
folder_path = "/home/docker/data/private/AuroraData/omni_real_data/omni_1min/2005" 
output_dir = "/home/docker/data/private/AuroraData/omni_real_data/omni_1min_pro/2005"
os.makedirs(output_dir, exist_ok=True)
for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith('.cdf'):
            full_path = os.path.join(root, file)
            print(f"找到 .cdf 文件: {full_path}")
        cdf_file = full_path
        #print(cdf_file)
        extract_core_omni(cdf_file, output_dir)

找到 .cdf 文件: /home/docker/data/private/AuroraData/omni_real_data/omni_1min/2005/omni_hro_1min_20050601_v01.cdf
处理文件: omni_hro_1min_20050601_v01
提取到日期: 20050601
应用数据范围过滤...
Bx: 过滤掉 2773 个异常点 (6.42%)
By: 过滤掉 2773 个异常点 (6.42%)
Bz: 过滤掉 2773 个异常点 (6.42%)
V: 过滤掉 7239 个异常点 (16.76%)
Vx: 过滤掉 7239 个异常点 (16.76%)
Vy: 过滤掉 7239 个异常点 (16.76%)
Vz: 过滤掉 7239 个异常点 (16.76%)
P: 过滤掉 7239 个异常点 (16.76%)

应用线性插值...

=== 插值效果 ===
Bx: 填充了 2773 个NaN值，剩余 0 个NaN
By: 填充了 2773 个NaN值，剩余 0 个NaN
Bz: 填充了 2773 个NaN值，剩余 0 个NaN
V: 填充了 7239 个NaN值，剩余 0 个NaN
Vx: 填充了 7239 个NaN值，剩余 0 个NaN
Vy: 填充了 7239 个NaN值，剩余 0 个NaN
Vz: 填充了 7239 个NaN值，剩余 0 个NaN
P: 填充了 7239 个NaN值，剩余 0 个NaN
已保存: /home/docker/data/private/AuroraData/omni_real_data/omni_1min_pro/2005/omni_20050601_1min.npy
找到 .cdf 文件: /home/docker/data/private/AuroraData/omni_real_data/omni_1min/2005/omni_hro_1min_20050801_v01.cdf
处理文件: omni_hro_1min_20050801_v01
提取到日期: 20050801
应用数据范围过滤...
Bx: 过滤掉 1624 个异常点 (3.64%)
By: 过滤掉 1624 个异常点 (3.64%)
Bz: 过滤掉 1624 个异常点 (3.64%)
V: 过滤掉 4842 个异常

In [2]:

# cd_file = cdflib.CDF("/home/docker/data/ro-share/omni/omni_cdaweb/hro_5min/1996/omni_hro_5min_19960101_v01.cdf")
# cdf_info_miu1 = cd_file.cdf_info()
# Var_miu1 = cdf_info_miu1.zVariables

# Vx = cd_file.varget('Vx')
# Vy = cd_file.varget('Vy')
# Vz = cd_file.varget('Vz')
# V = cd_file.varget('flow_speed')
# Epoch = cd_file.varget('Epoch')
# time = cdflib.epochs.CDFepoch.to_datetime(Epoch)

from datetime import timedelta
import numpy as np
import pandas as pd

# data_path = "/home/docker/data/private/AuroraData/generated_aurora_data/1996_omni_aurora/aurora_img_19960101.npy"
# df1 = pd.DataFrame(np.load(data_path,allow_pickle=True))
# time = df1['utc']
# t1 = time[0]
# print(type(t1))
# t2 = t1 + timedelta(minutes=55)
# print(t2)

aurora_path = "/home/docker/data/private/AuroraData/generated_aurora_data/1996_omni_aurora/aurora_img_19961201.npy"
aurora_data = np.load(aurora_path, allow_pickle=True)
print(aurora_data.shape)  # 形状: (时间点数量, 80, 96)
# # 选择第一个时间点的图像（或其他特定时间点）
# single_image = aurora_data[0]  # 形状: (80, 96)
# df2 = pd.DataFrame(single_image)


(8928, 80, 96)


In [1]:
import cdflib
omni_cdf = cdflib.CDF("/home/docker/data/private/AuroraData/omni_real_data/omni_1min/1996/omni_hro_1min_19960401_v01.cdf")

omni_info = omni_cdf.cdf_info()
zvariables = omni_info.zVariables
rVariables = omni_info.rVariables

In [3]:
import xarray as xr

file_path = "/home/docker/data/private/AuroraData/real_aurora_data/dmspf16_ssusi_edr-aurora_2005001T010906-2005001T025057-REV06220_vA8.2.0r000.nc"
data = xr.open_dataset(file_path)
# print(data)
# print(list(data.data_vars))

# 2. 选择半球（以北半球为例）
# 注意：能量通量图和地磁坐标网格都有明确的北半球变量
energy_flux_map = data['ENERGY_FLUX_NORTH_MAP']          # 形状可能为 (n, m)
geo_lat_grid = data['MODEL_NORTH_GEOGRAPHIC_LATITUDE']   # 形状应与上行一致
geo_lon_grid = data['MODEL_NORTH_GEOGRAPHIC_LONGITUDE']  # 形状应与上行一致
mlat_grid = data['LATITUDE_GEOMAGNETIC_GRID_MAP']
mlon_grid = data['LONGITUDE_GEOMAGNETIC_NORTH_GRID_MAP']
mlt_grid = data['MLT_GRID_MAP']
altitude = data['ALTITUDE']  # 可能是一个标量或一维数组
obs_time = data['TIME']  
data_quality = data['NORTH_DATA_QUALITY']

# 4. 检查并确保网格形状匹配（关键步骤）
print(f"能量通量图形状: {energy_flux_map.shape}")
print(f"地理纬度网格形状: {geo_lat_grid.shape}")
print(f"地理经度网格形状: {geo_lon_grid.shape}")
print(f"地磁纬度网格形状: {mlat_grid.shape}")
print(f"地磁经度网格形状: {mlon_grid.shape}")
print(f"地磁时角网格形状: {mlt_grid.shape}")
print(obs_time)


能量通量图形状: (363, 363)
地理纬度网格形状: (457,)
地理经度网格形状: (457,)
地磁纬度网格形状: (363, 363)
地磁经度网格形状: (363, 363)
地磁时角网格形状: (363, 363)
<xarray.DataArray 'TIME' ()> Size: 8B
[1 values with dtype=float64]
Attributes:
    TITLE:    Average time of scans in image
    UNITS:    Seconds since the start of the day
